# 03 — Model Experiments

**Purpose:** perform rolling temporal validation, tune CatBoost on a representative lifecycle configuration, train checkpoint/horizon model families, and evaluate untouched chronological holdouts.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)


Project root: C:\Users\BaneleS\Downloads\ml_refactor_final


In [2]:
import inspect

from game_performance.config import SETTINGS
from game_performance.io import load_artifact, save_artifact
from game_performance.modeling import train_all_model_families

SETTINGS.ensure_directories()

historical_snapshots = load_artifact(SETTINGS.feature_dir / "historical_snapshots.joblib")
print("Loaded configurations:", len(historical_snapshots))

# Show what is available before production training starts.
for key, snapshot in historical_snapshots.items():
    checkpoint, horizon = key
    print(f"checkpoint={checkpoint:>2}, horizon={horizon:>2}, rows={len(snapshot):,}")

Loaded configurations: 10
checkpoint= 7, horizon= 7, rows=9,803
checkpoint= 7, horizon=14, rows=9,609
checkpoint= 7, horizon=30, rows=9,532
checkpoint=14, horizon= 7, rows=8,783
checkpoint=14, horizon=14, rows=9,301
checkpoint=14, horizon=30, rows=9,142
checkpoint=14, horizon=60, rows=8,301
checkpoint=21, horizon=14, rows=8,607
checkpoint=21, horizon=30, rows=8,674
checkpoint=21, horizon=60, rows=7,876


In [3]:
# Production training: no max_families limit, all targets, regression + classification.
# This cell is intentionally compatible with both the original modeling.py and the improved modeling.py.

signature = inspect.signature(train_all_model_families)
production_kwargs = {}

if "mode" in signature.parameters:
    production_kwargs["mode"] = "production"

if "max_families" in signature.parameters:
    production_kwargs["max_families"] = None

if "run_regression" in signature.parameters:
    production_kwargs["run_regression"] = True

if "run_classification" in signature.parameters:
    production_kwargs["run_classification"] = True

print("Production training kwargs:", production_kwargs or "default full-production call")

training_results = train_all_model_families(
    historical_snapshots,
    SETTINGS,
    **production_kwargs,
)

print("Trained model families:", len(training_results["model_bundles"]))


Production training kwargs: {'mode': 'production', 'max_families': None, 'run_regression': True, 'run_classification': True}
Training family day 7 -> horizon 7 [production] ...
Finished day 7 -> horizon 7 in 195.7s
Training family day 7 -> horizon 14 [production] ...
Finished day 7 -> horizon 14 in 353.9s
Training family day 7 -> horizon 30 [production] ...
Finished day 7 -> horizon 30 in 121.2s
Training family day 14 -> horizon 7 [production] ...
Finished day 14 -> horizon 7 in 212.6s
Training family day 14 -> horizon 14 [production] ...
Finished day 14 -> horizon 14 in 190.1s
Training family day 14 -> horizon 30 [production] ...
Finished day 14 -> horizon 30 in 109.8s
Training family day 14 -> horizon 60 [production] ...
Finished day 14 -> horizon 60 in 108.4s
Training family day 21 -> horizon 14 [production] ...
Finished day 21 -> horizon 14 in 191.2s
Training family day 21 -> horizon 30 [production] ...
Finished day 21 -> horizon 30 in 91.0s
Training family day 21 -> horizon 60 [pr

In [4]:
holdout = training_results.get("holdout_predictions")
skipped = training_results.get("skipped_configurations")

print("Regression evaluation:", training_results["regression_evaluation"].shape)
print("Classification evaluation:", training_results["classification_evaluation"].shape)
print("Holdout predictions:", holdout.shape if holdout is not None else None)

if holdout is not None and not holdout.empty:
    print("\nTrained holdout model families:")
    display(
        holdout.groupby(["Obs_Window", "Horizon"])
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["Obs_Window", "Horizon"])
    )

print("\nSkipped configurations:")
if skipped is not None and not skipped.empty:
    display(skipped)
else:
    print("None")


Regression evaluation: (50, 14)
Classification evaluation: (10, 14)
Holdout predictions: (3165, 20)

Trained holdout model families:


,Obs_Window,Horizon,rows
0,7,7,372
1,7,14,384
2,7,30,408
3,14,7,369
4,14,14,498
5,14,30,311
6,14,60,129
7,21,14,385
8,21,30,193
9,21,60,116



Skipped configurations:
None


In [5]:
display(training_results["regression_evaluation"])
display(training_results["classification_evaluation"])

,Checkpoint,Horizon,Target,Holdout_Start,Best_Iteration,depth,learning_rate,l2_leaf_reg,random_strength,bagging_temperature,MAE,RMSE,R2,WAPE
0,7,7,target_clients,2026-06-22,553,7,0.03,8,0.5,1.0,3.184891e+03,1.264314e+04,0.757361,0.480035
1,7,7,target_betcount,2026-06-22,240,7,0.03,8,0.5,1.0,4.155149e+05,2.314205e+06,0.506129,0.616239
2,7,7,target_stakes,2026-06-22,369,5,0.05,5,1.0,0.5,6.821104e+05,2.711147e+06,0.813750,0.478145
3,7,7,target_ggr,2026-06-22,174,6,0.04,6,1.0,1.0,5.919406e+04,2.254295e+05,0.463036,0.817377
4,7,7,target_ggr_pct,2026-06-22,60,5,0.05,5,1.0,0.5,1.518434e-01,7.067876e-01,-0.038525,0.785161
5,7,14,target_clients,2026-06-14,891,7,0.03,8,0.5,1.0,4.322428e+03,1.581486e+04,0.855511,0.439018
6,7,14,target_betcount,2026-06-14,518,7,0.03,8,0.5,1.0,5.135322e+05,2.630159e+06,0.744284,0.527972
7,7,14,target_stakes,2026-06-14,551,5,0.05,5,1.0,0.5,2.167678e+06,6.027863e+06,0.644566,0.923810
8,7,14,target_ggr,2026-06-14,488,6,0.04,6,1.0,1.0,8.285711e+04,3.486775e+05,0.525784,0.697706
9,7,14,target_ggr_pct,2026-06-14,238,5,0.05,5,1.0,0.5,1.407155e-01,5.323426e-01,-0.161684,0.907885


,Checkpoint,Horizon,Holdout_Start,Best_Iteration,Accuracy,Precision,Recall,F1,ROC_AUC,depth,learning_rate,l2_leaf_reg,random_strength,bagging_temperature
0,7,7,2026-06-22,78,0.739247,0.574163,0.937500,0.712166,0.910252,5,0.05,5,1.0,0.5
1,7,14,2026-06-14,675,0.817708,0.653409,0.927419,0.766667,0.932041,5,0.05,5,1.0,0.5
2,7,30,2026-05-31,351,0.877451,0.741007,0.880342,0.804688,0.944077,5,0.05,5,1.0,0.5
3,14,7,2026-06-14,66,0.810298,0.601351,0.890000,0.717742,0.943271,5,0.05,5,1.0,0.5
4,14,14,2026-06-08,152,0.835341,0.625000,0.946970,0.753012,0.950758,5,0.05,5,1.0,0.5
5,14,30,2026-05-25,463,0.864952,0.709677,0.936170,0.807339,0.950632,5,0.05,5,1.0,0.5
6,14,60,2026-04-29,178,0.930233,0.823529,0.903226,0.861538,0.957538,5,0.05,5,1.0,0.5
7,21,14,2026-06-02,310,0.864935,0.648438,0.922222,0.761468,0.956874,5,0.05,5,1.0,0.5
8,21,30,2026-05-19,228,0.808290,0.655914,0.924242,0.767296,0.913386,5,0.05,5,1.0,0.5
9,21,60,2026-04-24,251,0.793103,0.607843,0.885714,0.720930,0.939683,5,0.05,5,1.0,0.5


In [6]:
reg = training_results["regression_evaluation"]

if not reg.empty:
    best_horizons = (
        reg.sort_values(["Checkpoint", "Target", "WAPE", "MAE"])
        .groupby(["Checkpoint", "Target"], as_index=False)
        .first()
    )
    display(best_horizons)
else:
    print("No regression evaluation rows were produced.")


,Checkpoint,Target,Horizon,Holdout_Start,Best_Iteration,depth,learning_rate,l2_leaf_reg,random_strength,bagging_temperature,MAE,RMSE,R2,WAPE
0,7,target_betcount,14,2026-06-14,518,7,0.03,8,0.5,1.0,5.135322e+05,2.630159e+06,0.744284,0.527972
1,7,target_clients,14,2026-06-14,891,7,0.03,8,0.5,1.0,4.322428e+03,1.581486e+04,0.855511,0.439018
2,7,target_ggr,14,2026-06-14,488,6,0.04,6,1.0,1.0,8.285711e+04,3.486775e+05,0.525784,0.697706
3,7,target_ggr_pct,7,2026-06-22,60,5,0.05,5,1.0,0.5,1.518434e-01,7.067876e-01,-0.038525,0.785161
4,7,target_stakes,7,2026-06-22,369,5,0.05,5,1.0,0.5,6.821104e+05,2.711147e+06,0.813750,0.478145
5,14,target_betcount,7,2026-06-14,249,7,0.03,8,0.5,1.0,1.619143e+05,1.025595e+06,0.758992,0.438961
6,14,target_clients,7,2026-06-14,462,7,0.03,8,0.5,1.0,1.989290e+03,6.634567e+03,0.861661,0.507813
7,14,target_ggr,7,2026-06-14,359,6,0.04,6,1.0,1.0,2.804081e+04,1.214216e+05,0.754378,0.509826
8,14,target_ggr_pct,7,2026-06-14,281,5,0.05,5,1.0,0.5,1.176540e-01,4.825323e-01,-0.180018,0.788727
9,14,target_stakes,7,2026-06-14,361,5,0.05,5,1.0,0.5,4.170060e+05,1.983512e+06,0.773334,0.435703


In [7]:
save_artifact(training_results, SETTINGS.model_dir / "training_results.joblib")
print("Saved production training results:", SETTINGS.model_dir / "training_results.joblib")


Saved production training results: C:\Users\BaneleS\Downloads\ml_refactor_final\artifacts\models\training_results.joblib
